In [3]:
import sys, json
sys.path.append("used-car-price-predictor")
import numpy as np
import pandas as pd
import joblib
from sklearn.base import clone
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import (GroupKFold, RandomizedSearchCV, GridSearchCV,
                                     cross_validate, cross_val_predict)
from xgboost import XGBRegressor
from src.features import FEATURES, make_model

train = pd.read_csv("used-car-price-predictor/data/processed/train.csv")
X_train, y_train, g_train = train[FEATURES], train["selling_price"], train["group_id"]
cv = GroupKFold(n_splits=5)          # same folds as Notebook 05, so comparisons are fair

results = pd.read_csv("used-car-price-predictor/reports/model_results/baseline_comparison.csv")
metric_cols = ["MAE (lakh)", "RMSE (lakh)", "MedAE (lakh)", "MAPE %", "R2 (CV)", "R2 (train)"]

In [4]:
tunable = ["Decision Tree", "Random Forest", "Gradient Boosting", "XGBoost"]
top2 = (results[results["model"].isin(tunable)]
        .sort_values("MAE (lakh)").head(2)["model"].tolist())
print("Candidates to tune:", top2)

Candidates to tune: ['XGBoost', 'Random Forest']


In [5]:
P = "regressor__model__"     # how sklearn addresses the model inside our pipeline

spaces = {
    "Decision Tree": {P+"max_depth": [4, 6, 8, 10, 14],
                      P+"min_samples_leaf": [1, 3, 5, 10],
                      P+"min_samples_split": [2, 5, 10]},
    "Random Forest": {P+"n_estimators": [200, 400, 600],
                      P+"max_depth": [None, 10, 15, 20],
                      P+"min_samples_leaf": [1, 2, 4],
                      P+"max_features": ["sqrt", 0.5, 1.0]},
    "Gradient Boosting": {P+"n_estimators": [200, 400, 600],
                          P+"learning_rate": [0.03, 0.05, 0.1],
                          P+"max_depth": [3, 4, 5],
                          P+"subsample": [0.7, 0.85, 1.0],
                          P+"min_samples_leaf": [1, 3, 5]},
    "XGBoost": {P+"n_estimators": [300, 500, 800],
                P+"learning_rate": [0.02, 0.05, 0.1],
                P+"max_depth": [4, 6, 8],
                P+"subsample": [0.7, 0.85, 1.0],
                P+"colsample_bytree": [0.6, 0.8, 1.0],
                P+"min_child_weight": [1, 3, 5],
                P+"reg_lambda": [1, 3, 10]},
}

bases = {   # n_jobs=1 inside the model, because the search itself runs in parallel
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42, n_jobs=1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "XGBoost": XGBRegressor(random_state=42, n_jobs=1),
}

In [6]:
N_ITER = 25
searches = {}
for name in top2:
    rs = RandomizedSearchCV(make_model(bases[name]), spaces[name], n_iter=N_ITER, cv=cv,
                            scoring="neg_mean_absolute_error", random_state=42,
                            n_jobs=-1, refit=True)
    rs.fit(X_train, y_train, groups=g_train)
    searches[name] = rs
    print(f"\n{name}: best CV MAE = {-rs.best_score_ / 1e5:.3f} lakh")
    print({k.replace(P, ""): v for k, v in rs.best_params_.items()})


XGBoost: best CV MAE = 0.719 lakh
{'subsample': 1.0, 'reg_lambda': 1, 'n_estimators': 500, 'min_child_weight': 1, 'max_depth': 4, 'learning_rate': 0.1, 'colsample_bytree': 0.8}

Random Forest: best CV MAE = 0.755 lakh
{'n_estimators': 400, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 15}


In [7]:
scoring = {"mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error",
           "medae": "neg_median_absolute_error",
           "mape": "neg_mean_absolute_percentage_error", "r2": "r2"}

def cv_metrics(model):
    res = cross_validate(model, X_train, y_train, groups=g_train, cv=cv,
                         scoring=scoring, return_train_score=True, n_jobs=-1)
    return {"MAE (lakh)": -res["test_mae"].mean() / 1e5,
            "RMSE (lakh)": -res["test_rmse"].mean() / 1e5,
            "MedAE (lakh)": -res["test_medae"].mean() / 1e5,
            "MAPE %": -res["test_mape"].mean() * 100,
            "R2 (CV)": res["test_r2"].mean(),
            "R2 (train)": res["train_r2"].mean()}

rows = []
for name in top2:
    before = results.loc[results["model"] == name].iloc[0][metric_cols].to_dict()
    rows.append({"model": name, "stage": "1. Untuned", **before})
    rows.append({"model": name, "stage": "2. RandomizedSearchCV", **cv_metrics(searches[name].best_estimator_)})

In [8]:
best_name = max(searches, key=lambda n: searches[n].best_score_)
rs_best = searches[best_name]

refine = {"Decision Tree": ["max_depth", "min_samples_leaf"],
          "Random Forest": ["max_depth", "min_samples_leaf"],
          "Gradient Boosting": ["max_depth", "learning_rate"],
          "XGBoost": ["max_depth", "learning_rate"]}

def local_grid(best_params, space, names):
    """For each chosen parameter: the best value and its neighbours in the original list."""
    grid = {}
    for n in names:
        choices = space[P + n]
        i = choices.index(best_params[P + n])
        grid[P + n] = choices[max(i - 1, 0): i + 2]
    return grid

grid = local_grid(rs_best.best_params_, spaces[best_name], refine[best_name])
print("Best model:", best_name, "| grid:", {k.replace(P, ""): v for k, v in grid.items()})

gs = GridSearchCV(clone(rs_best.best_estimator_), grid, cv=cv,
                  scoring="neg_mean_absolute_error", n_jobs=-1, refit=True)
gs.fit(X_train, y_train, groups=g_train)
print("Grid best CV MAE:", round(-gs.best_score_ / 1e5, 3), "lakh")
print({k.replace(P, ""): v for k, v in gs.best_params_.items()})

rows.append({"model": best_name, "stage": "3. GridSearchCV", **cv_metrics(gs.best_estimator_)})

Best model: XGBoost | grid: {'max_depth': [4, 6], 'learning_rate': [0.05, 0.1]}
Grid best CV MAE: 0.719 lakh
{'learning_rate': 0.1, 'max_depth': 4}


In [10]:
tuning = pd.DataFrame(rows)
tuning["Overfit gap (R2)"] = tuning["R2 (train)"] - tuning["R2 (CV)"]
tuning = tuning.round(3)
tuning.to_csv("used-car-price-predictor/reports/model_results/tuning_comparison.csv", index=False)
tuning

,model,stage,MAE (lakh),RMSE (lakh),MedAE (lakh),MAPE %,R2 (CV),R2 (train),Overfit gap (R2)
0,XGBoost,1. Untuned,0.724,1.628,0.416,15.597,0.892,0.977,0.085
1,XGBoost,2. RandomizedSearchCV,0.719,1.595,0.419,15.586,0.897,0.974,0.077
2,Random Forest,1. Untuned,0.760,1.618,0.444,16.734,0.893,0.978,0.085
3,Random Forest,2. RandomizedSearchCV,0.755,1.676,0.431,16.305,0.886,0.973,0.087
4,XGBoost,3. GridSearchCV,0.719,1.595,0.419,15.586,0.897,0.974,0.077


In [11]:
final_model = gs.best_estimator_ if gs.best_score_ > rs_best.best_score_ else rs_best.best_estimator_
final_params = gs.best_params_ if gs.best_score_ > rs_best.best_score_ else rs_best.best_params_
print("Final model:", best_name)

Final model: XGBoost


In [13]:
from sklearn.metrics import (mean_absolute_error, mean_squared_error, median_absolute_error,
                             mean_absolute_percentage_error, r2_score)

test = pd.read_csv("used-car-price-predictor/data/processed/test.csv")
X_test, y_test = test[FEATURES], test["selling_price"]
pred = final_model.predict(X_test)

test_scores = {"MAE (lakh)": mean_absolute_error(y_test, pred) / 1e5,
               "RMSE (lakh)": np.sqrt(mean_squared_error(y_test, pred)) / 1e5,
               "MedAE (lakh)": median_absolute_error(y_test, pred) / 1e5,
               "MAPE %": mean_absolute_percentage_error(y_test, pred) * 100,
               "R2": r2_score(y_test, pred)}
print(pd.Series(test_scores).round(3))
print("\nFor comparison, CV MAE on training data:",
      tuning[tuning["stage"] == "3. GridSearchCV"]["MAE (lakh)"].values)

MAE (lakh)       0.732
RMSE (lakh)      1.453
MedAE (lakh)     0.418
MAPE %          15.603
R2               0.924
dtype: float64

For comparison, CV MAE on training data: [0.719]


In [15]:
import os
os.makedirs("used-car-price-predictor/models", exist_ok=True)

joblib.dump(final_model, "used-car-price-predictor/models/best_model.joblib")

# Out-of-fold predictions on training data: used on Day 3 to build honest price ranges
oof = cross_val_predict(final_model, X_train, y_train, groups=g_train, cv=cv, n_jobs=-1)
pd.DataFrame({"actual": y_train.values, "predicted": oof, "group_id": g_train.values}) \
  .to_csv("used-car-price-predictor/reports/model_results/oof_predictions.csv", index=False)

summary = {"model": best_name,
           "best_params": {k.replace(P, ""): v for k, v in final_params.items()},
           "test_scores": {k: round(float(v), 4) for k, v in test_scores.items()},
           "features": FEATURES}
json.dump(summary, open("used-car-price-predictor/models/model_summary.json", "w"), indent=2, default=str)
print("Saved model, OOF predictions and summary")

Saved model, OOF predictions and summary
